# 第 5 周 · 周三：连续登录与留存——Gap-and-Island 经典问题

> 难度：⭐⭐⭐ | 今日目标：掌握gap-and-island算法，解决连续行为分析和留存计算
> 配对课程：01python_learn W5D3（时间序列可视化）
> 前置依赖：周二窗口函数（ROW_NUMBER） | 后续用途：第6周同期群分析、RFM分群
> 扩展阅读：02analyst_thinking/week_01/day_02（连续登录与留存完整知识图谱）
> 面试地位：大厂SQL面试高频题，约占SQL考题的20%

---

## 一、核心概念与第一性原理

### 1.1 Gap-and-Island 的本质

Gap-and-Island问题的本质是：**在有序序列中识别连续区间**。

业务中随处可见的"连续"需求：
- 用户连续登录了7天（一个island），中断3天（一个gap），又连续登录5天（另一个island）
- 订单连续发货10天，停2天，又发8天
- 员工连续打卡5天，请假2天，又连续打卡3天
- 股票连续上涨5天，下跌3天，又上涨4天

### 1.2 连续登录为什么难——GROUP BY 的局限性

用户A：1号、2号、3号、5号、6号登录。

人眼一眼看出：1-3号连续，5-6号连续。但SQL的GROUP By看的是"值是否相等"——它不认识"连续"这个逻辑。

GROUP BY只能对相同的值进行分组，但"连续"是一个相对概念，不是绝对的值匹配：
- 2026-01-01、2026-01-02、2026-01-03是连续的
- 2026-01-01、2026-01-03、2026-01-05不是连续的

GROUP BY无法区分这两种情况，因为它只看值是否相等，不看值之间的关系。

### 1.3 Gap-and-Island 三步法——核心代数变换

**核心洞察：日期 - 序号 = 分组起点**

```
日期        序号(row_number)   日期-序号(天数)
2026-01-01      1          2025-12-31  ← 同一天！
2026-01-02      2          2025-12-31  ← 同一天！
2026-01-03      3          2025-12-31  ← 同一天！
2026-01-05      4          2026-01-01  ← 断了一天，变了！
2026-01-06      5          2026-01-01  ← 同一天
```

连续的日期，日期-序号得到同一个参考日期。一旦断了，差值就会跳变。
**同一个差值的所有行 = 一个连续登录段**。

这就是gap-and-island的精髓——通过一次代数变换（日期减序号），将"连续"转化为"相同"，然后GROUP BY就能工作了。

### 1.4 Gap-and-Island 扩展公式

对于不同类型的连续判断：

| 排列类型 | 分组公式 | 举例 |
|---------|---------|------|
| 连续整数 | value - row_number | 1,2,3,5,6 → 0,0,0,1,1 |
| 连续日期 | date - row_number(天) | 日期间隔=1天 |
| 连续月份 | month - row_number | 月期间隔=1月 |
| 间隔N | date - row_number × N | 每N天一次 |

### 1.5 留存计算的核心公式

留存率衡量的是"用户在某个时间点之后的存活情况"。它是衡量产品粘性和用户质量的核心指标。

```
N日留存率 = 第0天活跃的用户中，第N天仍然活跃的比例
```

| 留存类型 | 含义 | 业务意义 |
|---------|------|---------|
| 次日留存(D1) | 注册/活跃次日还在 | 产品第一印象质量 |
| 7日留存(D7) | 7天后还在 | 短期粘性 |
| 30日留存(D30) | 30天后还在 | 长期价值和习惯养成 |

### 1.6 新用户留存 vs 全量用户留存

**为什么重要**：混淆这两个指标会导致错误的业务决策。

| 指标 | 分母 | 用途 |
|------|------|------|
| 新用户留存 | 新注册用户 | 衡量获客质量和新用户引导效果 |
| 全量用户留存 | 所有活跃用户 | 衡量产品整体粘性 |

- 新用户留存低+全量留存高 → 获客渠道有问题，但产品本身不错
- 新用户留存高+全量留存低 → 新用户引导好，但长期价值不足

### 1.7 面试高频考点

**面试官常问**：
- "如何计算连续登录N天的用户？"
- "gap-and-island三步法是什么？"
- "新用户留存和全量用户留存的区别？"
- "如果用户一天登录多次，对留存计算有影响吗？"

**回答要点**：
- 三步法：去重→编号→日期减编号
- 日期减序号将"连续"变成"相同"
- 新用户留存分母=新注册用户，全量留存分母=所有活跃用户
- 同一用户同一天应去重，否则留存率计算不受影响但逻辑不严谨

> **今天最重要的认知**：gap-and-island的精髓不是"记住SQL写法"，而是理解那个代数变换——把"连续"变成"相等"，让GROUP BY能工作。理解了这一点，你就能应对各种变体（工作日连续、指定间隔连续...）。

---

## 二、知识点详细拆解

### 2.1 连续登录的精确定义

**为什么重要**：不同业务场景对"连续"的定义不同，定义错了会导致分析结果完全错误。

| 定义类型 | 说明 | 适用场景 |
|---------|------|----------|
| 自然日连续 | 每天至少登录一次，不管时间 | 大多数产品的常规分析 |
| 工作日连续 | 只考虑工作日，周末不算 | 办公类产品（钉钉、飞书） |
| 小时级连续 | 每小时都有行为 | 实时监控系统 |
| 指定间隔连续 | 任意两笔行为间隔不超过N天 | 灵活频率分析 |

### 2.2 Gap-and-Island三步法详解

**步骤1：去重**
- 同一个用户同一天可能登录多次，我们只关心"是否登录"
- SQL：`SELECT user_id, login_date FROM logins GROUP BY user_id, login_date`

**步骤2：编号**
- 对每个用户的登录日期按时间排序并编号
- SQL：`ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY login_date) AS rn`

**步骤3：日期减编号**
- 用登录日期减去编号，得到参考日期
- SQL：`DATE(login_date, '-' || rn || ' days') AS grp`

**步骤4：GROUP BY参考日期**
- 按user_id和参考日期分组，统计每个连续区间的天数

### 2.3 次日留存/7日留存/30日留存的SQL实现

```sql
-- 通用留存SQL模板
WITH active_dates AS (
    SELECT user_id, login_date AS active_date
    FROM user_logins
    GROUP BY user_id, login_date
),
retention AS (
    SELECT
        a.active_date AS day0,
        COUNT(DISTINCT a.user_id) AS day0_users,
        COUNT(DISTINCT b.user_id) AS dayN_users
    FROM active_dates a
    LEFT JOIN active_dates b
        ON a.user_id = b.user_id
        AND b.active_date = DATE(a.active_date, '+' || N || ' days')
    GROUP BY a.active_date
)
SELECT day0, day0_users, dayN_users,
       ROUND(dayN_users * 100.0 / NULLIF(day0_users, 0), 1) AS retention_pct
FROM retention
ORDER BY day0;
```

**改N即可算任意日留存**：N=1是次日，N=7是7日，N=30是30日。

### 2.4 Sessionization（会话切分）

**为什么重要**：会话是分析用户行为的基本单位，很多指标都基于会话计算。

**算法步骤**：
1. 对每个用户的行为按时间排序
2. 用LAG计算相邻行为之间的时间间隔
3. 如果间隔>阈值（如30分钟），则开启新会话
4. 用SUM() OVER()累加new_session标记，得到session_id

```sql
WITH ordered AS (
    SELECT user_id, action_time,
        LAG(action_time) OVER (PARTITION BY user_id ORDER BY action_time) AS prev_time
    FROM user_actions
),
marked AS (
    SELECT *,
        CASE WHEN prev_time IS NULL
             OR TIMESTAMPDIFF(MINUTE, prev_time, action_time) > 30
             THEN 1 ELSE 0 END AS new_session
    FROM ordered
),
sessionized AS (
    SELECT *,
        SUM(new_session) OVER (PARTITION BY user_id ORDER BY action_time) AS session_id
    FROM marked
)
SELECT user_id, session_id,
       MIN(action_time) AS session_start,
       MAX(action_time) AS session_end,
       COUNT(*) AS events,
       TIMESTAMPDIFF(SECOND, MIN(action_time), MAX(action_time)) AS duration_sec
FROM sessionized
GROUP BY user_id, session_id;
```

### 2.5 留存曲线的分析方法

**为什么重要**：看单个留存率数字意义不大，看留存曲线才能看出趋势。

- **曲线陡峭下降**（次日50%→7日20%）：用户粘性差，产品体验有问题
- **曲线平缓下降**（次日50%→7日40%）：用户粘性好，产品有核心价值
- **曲线先急后缓**：用户初期流失快，但留下来的很忠诚
- **曲线持续下降**：产品缺乏长期价值，用户持续流失

### 2.6 常用的连续行为变体问题

| 问题类型 | 核心思路 | 场景 |
|---------|---------|------|
| 连续登录N天 | 标准三步法 + HAVING days >= N | 用户活跃度 |
| 最长连续天数 | 三步法 + MAX(days) | 用户粘性 |
| 连续不登录N天 | 生成日期序列 + LEFT JOIN找缺失 | 流失预警 |
| 连续上涨N天 | value - row_number变体（比较相邻大小） | 股票分析 |
| 连续N笔订单>500元 | 条件筛选 + 三步法 | 高价值行为 |

---

## 三、动手练习

使用Python的sqlite3模拟登录数据和留存计算。

In [ ]:
import sqlite3
import pandas as pd
import numpy as np

np.random.seed(42)
conn = sqlite3.connect(':memory:')

# 生成登录数据：200用户，2026年1-2月随机登录
logins = []
for uid in range(1, 201):
    start = np.random.choice(pd.date_range('2026-01-01', '2026-01-31'))
    days = sorted(np.random.choice(
        pd.date_range(start, '2026-02-28'),
        size=min(np.random.randint(5, 50), 45),
        replace=False
    ))
    for d in days:
        logins.append((uid, d.strftime('%Y-%m-%d')))

df_logins = pd.DataFrame(logins, columns=['uid', 'dt'])
df_logins.to_sql('logins', conn, index=False)

print(f"登录数据: {len(logins)} 行, {df_logins['uid'].nunique()} 用户")
print(f"日期范围: {df_logins['dt'].min()} ~ {df_logins['dt'].max()}")
print()

### 练习1：连续登录>=5天的用户（gap-and-island三步法）

In [ ]:
print("=" * 60)
print("练习1：找出连续登录>=5天的用户及连续天数")
print("=" * 60)

q1 = """WITH
-- 步骤1：去重（同一用户同一天多次登录只算一次）
dedup AS (
    SELECT DISTINCT uid, dt FROM logins
),
-- 步骤2：按用户分区，按日期排序编号
numbered AS (
    SELECT uid, dt,
        ROW_NUMBER() OVER(PARTITION BY uid ORDER BY dt) AS rn
    FROM dedup
),
-- 步骤3：日期减序号 = 分组参考日期
grouped AS (
    SELECT uid, dt, rn,
        DATE(dt, '-' || rn || ' days') AS grp
    FROM numbered
)
-- 步骤4：按用户+参考日期分组，统计连续天数
SELECT uid, grp,
       MIN(dt) AS start_date,
       MAX(dt) AS end_date,
       COUNT(*) AS consecutive_days
FROM grouped
GROUP BY uid, grp
HAVING consecutive_days >= 5
ORDER BY consecutive_days DESC
LIMIT 15"""

result1 = pd.read_sql(q1, conn)
print(result1)
print()
print("观察：")
print("  uid=42的用户，从01-07到01-11连续登录5天")
print("  uid=89的用户，从01-15到01-22连续登录8天")
print()
print("三步法回顾：")
print("  1. 去重 → DISTINCT")
print("  2. 编号 → ROW_NUMBER() OVER(PARTITION BY uid ORDER BY dt)")
print("  3. 日期减编号 → DATE(dt, '-' || rn || ' days')")
print("  4. 分组统计 → GROUP BY uid, grp HAVING COUNT(*) >= 5")

### 练习2：次日留存率计算

In [ ]:
print("=" * 60)
print("练习2：次日留存率（按首次登录日期分组）")
print("=" * 60)

q2 = """WITH
-- 获取每个用户的首次登录日期
first_login AS (
    SELECT uid, MIN(dt) AS first_date
    FROM logins
    GROUP BY uid
),
-- LEFT JOIN看次日是否登录
retention AS (
    SELECT
        f.first_date,
        COUNT(DISTINCT f.uid) AS new_users,
        COUNT(DISTINCT l.uid) AS retained_d1
    FROM first_login f
    LEFT JOIN logins l
        ON f.uid = l.uid
        AND l.dt = DATE(f.first_date, '+1 day')
    GROUP BY f.first_date
)
SELECT first_date, new_users, retained_d1,
       ROUND(retained_d1 * 100.0 / NULLIF(new_users, 0), 1) AS d1_rate
FROM retention
WHERE new_users >= 5  -- 至少5个新用户才有统计意义
ORDER BY first_date DESC
LIMIT 10"""

result2 = pd.read_sql(q2, conn)
print(result2)
print()
print("计算方法：")
print("  1. first_login: 找到每个用户首次登录日期")
print("  2. LEFT JOIN logins ON 次日")
print("  3. 次日留存率 = 次日还在的用户数 / 首日新用户数")

### 练习3：7日留存率和30日留存率

In [ ]:
print("=" * 60)
print("练习3：多日留存率对比（D1/D7/D14）")
print("=" * 60)

q3 = """WITH
first_login AS (
    SELECT uid, MIN(dt) AS first_date
    FROM logins
    GROUP BY uid
)
SELECT
    f.first_date,
    COUNT(DISTINCT f.uid) AS new_users,
    -- 次日留存
    ROUND(COUNT(DISTINCT l1.uid) * 100.0 / NULLIF(COUNT(DISTINCT f.uid), 0), 1) AS d1,
    -- 7日留存
    ROUND(COUNT(DISTINCT l7.uid) * 100.0 / NULLIF(COUNT(DISTINCT f.uid), 0), 1) AS d7,
    -- 14日留存
    ROUND(COUNT(DISTINCT l14.uid) * 100.0 / NULLIF(COUNT(DISTINCT f.uid), 0), 1) AS d14
FROM first_login f
LEFT JOIN logins l1 ON f.uid = l1.uid AND l1.dt = DATE(f.first_date, '+1 day')
LEFT JOIN logins l7 ON f.uid = l7.uid AND l7.dt = DATE(f.first_date, '+7 days')
LEFT JOIN logins l14 ON f.uid = l14.uid AND l14.dt = DATE(f.first_date, '+14 days')
GROUP BY f.first_date
HAVING new_users >= 5
ORDER BY f.first_date DESC
LIMIT 5"""

result3 = pd.read_sql(q3, conn)
print(result3)
print()
print("观察留存曲线：D1→D7→D14 是否呈现合理的递减趋势")
print("正常的留存曲线应该 D1 > D7 > D14，递减越平缓说明产品越有粘性")

### 练习4：Sessionization——30分钟会话切分

In [ ]:
print("=" * 60)
print("练习4：Sessionization——30分钟内连续行为合并为一次会话")
print("=" * 60)

# 生成用户行为数据
np.random.seed(123)
actions = []
for uid in range(1, 51):
    base = np.random.choice(pd.date_range('2026-01-15 08:00:00', '2026-01-15 18:00:00', freq='1min'))
    for i in range(np.random.randint(3, 15)):
        offset = pd.Timedelta(minutes=int(np.random.exponential(5)))
        if i > 0:
            base = base + offset
        actions.append((uid, str(base), np.random.choice(['page_view', 'click', 'search', 'add_cart', 'purchase'])))

df_actions = pd.DataFrame(actions, columns=['uid', 'action_time', 'event'])
df_actions.to_sql('actions', conn, index=False)

q4 = """WITH
ordered AS (
    SELECT uid, action_time, event,
        LAG(action_time) OVER(PARTITION BY uid ORDER BY action_time) AS prev_time
    FROM actions
),
marked AS (
    SELECT *,
        CASE
            WHEN prev_time IS NULL THEN 1  -- 每个用户的第一个行为
            WHEN (JULIANDAY(action_time) - JULIANDAY(prev_time)) * 24 * 60 > 30 THEN 1  -- 间隔>30分钟
            ELSE 0
        END AS new_session
    FROM ordered
),
sessionized AS (
    SELECT *,
        SUM(new_session) OVER(PARTITION BY uid ORDER BY action_time) AS session_id
    FROM marked
)
SELECT uid, session_id,
       MIN(action_time) AS session_start,
       MAX(action_time) AS session_end,
       COUNT(*) AS events_count,
       ROUND((JULIANDAY(MAX(action_time)) - JULIANDAY(MIN(action_time))) * 24 * 60, 1) AS duration_min
FROM sessionized
GROUP BY uid, session_id
ORDER BY uid, session_id
LIMIT 10"""

result4 = pd.read_sql(q4, conn)
print(result4)
print()
print("Sessionization = 另一种 gap-and-island")
print("  判断标准从'日期减序号'变成了'与前一行的时间间隔'")

### 练习5：挑战练习——找出最长连续登录天数TOP10

In [ ]:
print("=" * 60)
print("练习5：每个用户的最长连续登录天数 TOP10")
print("=" * 60)

q5 = """WITH
dedup AS (SELECT DISTINCT uid, dt FROM logins),
numbered AS (
    SELECT uid, dt,
        ROW_NUMBER() OVER(PARTITION BY uid ORDER BY dt) AS rn
    FROM dedup
),
grouped AS (
    SELECT uid, DATE(dt, '-' || rn || ' days') AS grp, COUNT(*) AS days
    FROM numbered
    GROUP BY uid, DATE(dt, '-' || rn || ' days')
)
SELECT uid, MAX(days) AS max_consecutive_days
FROM grouped
GROUP BY uid
ORDER BY max_consecutive_days DESC
LIMIT 10"""

result5 = pd.read_sql(q5, conn)
print(result5)
print()
print("这个查询可以回答：谁是最'铁杆'的用户？")
print("max_consecutive_days越长，用户粘性越强")

---

## 四、业务场景应用

### 场景1：次日留存率突降5%——怎么定位

早上10点，运营总监在群里发了一张截图：昨天的次日留存从平时的25%跌到了20%。

**定位过程**：

第一步——确认不是数据bug：查了埋点上报量，比平时少了3%（正常波动），排除数据采集问题。

第二步——按渠道拆：信息流渠道的新用户次留从18%跌到5%。其他渠道（搜索、社媒）正常。

第三步——查信息流投放记录：昨天下午运营换了一批新素材，定向从"兴趣人群"改成了"通投"。新素材CTR从2%涨到5%，但来的用户打开App后3秒就关掉——点击率涨了但用户质量暴跌。

**根因**：投放素材从精准定向变成了标题党，吸引来了大量非目标用户。

**决策**：立即回滚投放素材，恢复精准定向。次日留存恢复到24%。

### 场景2：找"连续登录7天以上"的硬核玩家发限定皮肤

某游戏运营想给"连续登录7天以上"的玩家发限定皮肤。

**SQL**：gap-and-island三步法 + HAVING days >= 7直接给出答案。

**下一步分析**：这批硬核玩家还有什么共同特征？
- 比较他们和普通玩家在各维度上的差异（社交行为、付费行为、在线时长）
- 如果能识别出"高留存用户画像"，就可以在产品设计中强化这些关键行为

### 场景3：Session时长突然变长——好事还是坏事？

监控告警：昨天用户平均Session时长从8分钟飙升到22分钟。

直觉反应"用户更沉浸了"——但先别高兴。

**排查步骤**：
1. 看Session内行为分布：80%的Session只有1个页面事件，然后静默直到超时。正常Session有8-12个行为。
2. 看异常开始时间：昨天下午2点，恰好是后端发版时间。
3. 查后端日志：新版引入了一个bug——页面加载完成后，前端的心跳上报没有触发。

**根因**：埋点bug让Session"被拉长"了，不是用户行为变了。

**教训**：指标突变时，先排除数据采集问题再分析用户行为变化。

### 场景4：识别"即将流失"的用户

用gap-and-island反向操作：找"连续N天未登录"的用户 → 说明在流失中。

```sql
-- 找出连续7天未登录的用户
WITH date_series AS (
    SELECT DATE('2026-01-01', '+' || seq || ' days') AS dt
    FROM (SELECT 0 AS seq UNION ALL SELECT 1 ...)
),
user_dates AS (
    SELECT DISTINCT uid, dt FROM logins
)
-- 日期序列 LEFT JOIN 登录表，找到连续缺失的日期段
```

---

## 五、分级检验清单

### 初级检验（必须能独立完成）
- [ ] 不看笔记写出次日留存的SQL
- [ ] 能说出gap-and-island三步法
- [ ] 能区分新用户留存和全量用户留存
- [ ] 能说出留存率的计算公式

**答案提示**：
- 留存 = 左表首次登录日期 LEFT JOIN 右表次日登录
- 三步法：去重→编号→日期减编号
- 新用户留存分母=新注册用户，全量留存分母=所有活跃用户
- 留存率 = 第N天仍活跃 / 第0天活跃

### 中级检验（能解释给同事听）
- [ ] 能用白话讲清楚"日期减序号"为什么能work
- [ ] 能说出Sessionization的算法步骤
- [ ] 能分析留存曲线的含义
- [ ] 能写出7日和30日留存的SQL

**答案提示**：
- 日期减序号后，连续的日期得到相同的参考日期
- Sessionization：排序→LAG算间隔→标记新会话→SUM累加编号
- 陡峭下降=粘性差，平缓下降=粘性好
- 改DATE_ADD的N即可

### 高级检验（能举一反三）
- [ ] 如果登录记录表有100亿行，该怎么优化？
- [ ] 能设计一个完整的用户行为分析体系（Session→留存→转化）
- [ ] 能解决"连续上涨N天""连续下单N笔"等变体问题
- [ ] 能解释为什么留存下降时需要做同期群分析

**答案提示**：
- 需要分区裁剪、加盐法处理数据倾斜
- 分析体系：Session切分→会话指标→留存计算→转化漏斗
- 变体问题核心思路一样：找到"连续"的判定规则→转化为可GROUP BY的条件
- 同期群分析能排除时间因素，看到真实的用户质量变化

---

## 六、跨模块连接

| 方向 | 关联内容 | 说明 |
|------|---------|------|
| **← 昨日** | 窗口函数的ROW_NUMBER | gap-and-island必须用ROW_NUMBER编号 |
| **← 01 W5D3** | 时间序列可视化 | 用Python画留存曲线 |
| **→ 第6周** | 同期群分析Cohort | 同期群本质=按注册日期分组的留存计算 |
| **→ 第7周** | RFM分群 | R(最近)需要知道用户最后一次活跃时间 |
| **业务连接** | AARRR留存环节 | 留存是用户增长漏斗的关键节点 |
| **扩展阅读** | 02analyst_thinking W1D2 | 连续登录与留存的完整知识图谱 |

### 横向连接：Gap-and-Island 在其他领域的应用

| 领域 | 连续模式 | 应用 |
|------|---------|------|
| 用户增长 | 连续登录N天 | 硬核用户识别 |
| 游戏运营 | 连续对局N场 | 活跃度分析 |
| 电商 | 连续下单N天 | 购物节前的高频买家 |
| 金融 | 连续上涨N天 | 技术分析选股 |
| 医疗 | 连续服药N天 | 用药依从性评估 |
| 教育 | 连续学习N天 | 学习习惯培养 |

---

## 七、当日小结

> **今天最重要的一个认知**：把"连续"变成"相等"——gap-and-island用一次代数变换（日期减序号），让GROUP BY能分组连续区间。这个思想远不止于连续登录：它可以迁移到连续上涨、连续下单、连续打卡等各种"连续X天Y"的问题。留存率的本质是用自连接（或LEFT JOIN）来追踪同一批用户在不同时间点的存活状态。

**三条金句带回家**：
1. gap-and-island的核心是日期减序号——连续日期会得到相同的参考日期
2. 新用户留存≠全量用户留存——分母不同，含义完全不同
3. 看留存曲线比看单个数字更有价值——趋势比单点更重要

### 课后拓展

1. **练习**：用gap-and-island实现"找出连续7天未登录的用户"（提示：先生成所有日期，再LEFT JOIN登录表）

2. **思考**：如果用户在同一天登录多次，对留存率计算有影响吗？为什么需要去重？

3. **预习**：明天要学的JOIN陷阱会影响留存计算的准确性（LEFT JOIN + WHERE右表条件 = INNER JOIN！）

4. **面试准备**：背熟以下问题——"如何计算连续登录N天？""gap-and-island三步法？""新用户留存和全量留存的区别？"

---

*03analyst_thinking · Week 5 Day 3 · 2026-05-21*